# PhishGuard AI — VS Code URL Model Training

Trains and evaluates URL-only phishing classifiers on the local PhiUSIIL CSV. No Google Colab features or webpage fetching are used. Place the CSV in `data/`, or set `DATASET_PATH` in Cell 2.

In [31]:
# Cell 1 — Imports and project setup
# If these packages are missing, run in a VS Code notebook cell: %pip install pandas numpy scikit-learn joblib
from pathlib import Path
import json
import os
import sys
from datetime import datetime, timezone

PROJECT_ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "app.py").is_file() or (path / ".git").exists()),
    Path.cwd(),
)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import GroupShuffleSplit
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier
from url_features import URL_FEATURES as SHARED_URL_FEATURES, extract_url_features

print("Project root:", PROJECT_ROOT)

Project root: c:\Users\vaishu\PhishGuard-AI


In [32]:
# Cell 2 — Find/load the PhiUSIIL CSV
# Set this to a CSV path if your dataset is stored somewhere else.
DATASET_PATH = Path(r"C:\Users\vaishu\Downloads\PhiUSIIL_Phishing_URL_Dataset.csv")
if DATASET_PATH and DATASET_PATH.is_file():
    DATASET_FILE = DATASET_PATH
else:
    search_roots = [PROJECT_ROOT, Path.home() / "Downloads"]
    candidates = []
    for search_root in search_roots:
        if search_root.exists():
            candidates.extend(
                path for path in search_root.rglob("*.csv")
                if ".venv" not in path.parts and "site-packages" not in path.parts
                and ("phiusiil" in path.name.lower()
                     or ("phishing" in path.name.lower() and "url" in path.name.lower()))
            )
    candidates = list(dict.fromkeys(candidates))
    if len(candidates) != 1:
        raise FileNotFoundError(
            "Could not uniquely locate the PhiUSIIL CSV. Set DATASET_PATH "
            "to its full path. Candidates: " + repr([str(path) for path in candidates])
        )
    DATASET_FILE = candidates[0]

df = pd.read_csv(DATASET_FILE, low_memory=False)
print("Dataset:", DATASET_FILE)
print("Dataset shape:", df.shape)

Dataset: C:\Users\vaishu\Downloads\PhiUSIIL_Phishing_URL_Dataset.csv
Dataset shape: (235795, 55)


In [33]:
# Cell 3 — Inspect and validate the dataset
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nMissing values by column:")
print(df.isna().sum().to_string())

required_columns = {"URL", "label"}
missing_metadata_columns = required_columns.difference(df.columns)
if missing_metadata_columns:
    raise ValueError(
        "Required dataset columns are missing: "
        + ", ".join(sorted(missing_metadata_columns))
    )
if df["label"].isna().any():
    raise ValueError("Target column 'label' contains missing values.")
target_values = set(df["label"].unique().tolist())
if target_values != {0, 1}:
    raise ValueError(
        f"Target 'label' must contain exactly 0 and 1; found {target_values}."
    )
if df["URL"].isna().any():
    raise ValueError("URL column contains missing values; fix these rows before extraction.")
print("\nFirst 5 URLs:")
print(df["URL"].head())
print("\nTarget distribution (0=phishing, 1=legitimate):")
print(df["label"].value_counts().sort_index())

Dataset shape: (235795, 55)

Column names:
['URL', 'URLLength', 'Domain', 'DomainLength', 'IsDomainIP', 'TLD', 'URLSimilarityIndex', 'CharContinuationRate', 'TLDLegitimateProb', 'URLCharProb', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS', 'LineOfCode', 'LargestLineLength', 'HasTitle', 'Title', 'DomainTitleMatchScore', 'URLTitleMatchScore', 'HasFavicon', 'Robots', 'IsResponsive', 'NoOfURLRedirect', 'NoOfSelfRedirect', 'HasDescription', 'NoOfPopup', 'NoOfiFrame', 'HasExternalFormSubmit', 'HasSocialNet', 'HasSubmitButton', 'HasHiddenFields', 'HasPasswordField', 'Bank', 'Pay', 'Crypto', 'HasCopyrightInfo', 'NoOfImage', 'NoOfCSS', 'NoOfJS', 'NoOfSelfRef', 'NoOfEmptyRef', 'NoOfExternalRef', 'label']

Missing values by column:
URL                    

In [34]:
# Cell 4 — Define the URL-only feature set in its shared training/API order
feature_columns = json.loads(FEATURE_COLUMNS_FILE.read_text(encoding="utf-8"))
if feature_columns != URL_FEATURES:
    raise ValueError(
        "url_feature_columns.json differs from the shared URL_FEATURES order. "
        "Regenerate it from url_features.py before training."
    )
if len(URL_FEATURES) != 19 or len(set(URL_FEATURES)) != 19:
    raise ValueError("Expected exactly 19 distinct URL-only features.")
if "label" in URL_FEATURES or "Domain" in URL_FEATURES:
    raise ValueError("Target/group columns must never be model input features.")
print("Verified exact shared feature order:", URL_FEATURES)

Verified exact shared feature order: ['URLLength', 'DomainLength', 'IsDomainIP', 'CharContinuationRate', 'TLDLength', 'NoOfSubDomain', 'HasObfuscation', 'NoOfObfuscatedChar', 'ObfuscationRatio', 'NoOfLettersInURL', 'LetterRatioInURL', 'NoOfDegitsInURL', 'DegitRatioInURL', 'NoOfEqualsInURL', 'NoOfQMarkInURL', 'NoOfAmpersandInURL', 'NoOfOtherSpecialCharsInURL', 'SpacialCharRatioInURL', 'IsHTTPS']


In [35]:
# Cell 5 — Smoke-test samples, capture baseline, and extract the raw URL column
sample_urls = [
    "https://www.google.com",
    "https://example.com",
    "http://secure-login-example.com",
]
sample_feature_rows = [extract_url_features(sample_url) for sample_url in sample_urls]
sample_feature_frame = pd.DataFrame(sample_feature_rows, columns=URL_FEATURES)
print("Sample URL features (before training):")
display(pd.concat([pd.Series(sample_urls, name="URL"), sample_feature_frame], axis=1))

# Capture probabilities from the existing model before replacing it.
baseline_probabilities = None
if MODEL_FILE.is_file() and MODEL_FILE.stat().st_size > 0:
    previous_model = joblib.load(MODEL_FILE)
    if previous_model.n_features_in_ != len(URL_FEATURES):
        raise ValueError("Existing model does not accept the current 19-feature input.")
    previous_probabilities = previous_model.predict_proba(sample_feature_frame)
    baseline_probabilities = {
        sample_url: {
            "phishing_probability": float(previous_probabilities[i, list(previous_model.classes_).index(0)]),
            "legitimate_probability": float(previous_probabilities[i, list(previous_model.classes_).index(1)]),
            "prediction": int(previous_model.predict(sample_feature_frame.iloc[[i]])[0]),
        }
        for i, sample_url in enumerate(sample_urls)
    }
    print("Captured current model probabilities for before/after comparison.")
else:
    print("No non-empty previous model found; after-training predictions will still be reported.")

try:
    from tqdm.auto import tqdm
except ImportError:
    tqdm = None

urls = df["URL"]
extracted_rows = []
if tqdm is not None:
    for url in tqdm(urls, total=len(urls), desc="Extracting URL features"):
        extracted_rows.append(extract_url_features(url))
else:
    for row_number, url in enumerate(urls, start=1):
        extracted_rows.append(extract_url_features(url))
        if row_number % 10000 == 0 or row_number == len(urls):
            print(f"Extracted {row_number:,}/{len(urls):,} URLs")

X = pd.DataFrame(extracted_rows, index=df.index, columns=URL_FEATURES)
y = pd.to_numeric(df["label"], errors="raise").astype(int)
if X.shape != (len(df), 19):
    raise ValueError(f"Unexpected extracted feature matrix shape: {X.shape}")
if X.columns.tolist() != URL_FEATURES:
    raise ValueError("Extracted feature order does not match URL_FEATURES.")
if X.isna().any().any():
    raise ValueError("Extracted URL features contain NaN values.")
if not np.isfinite(X.to_numpy(dtype=float)).all():
    raise ValueError("Extracted URL features contain infinite values.")

print("Successfully processed URLs:", len(X))
print("X.shape:", X.shape)
print("Feature order verified:", X.columns.tolist() == URL_FEATURES)
print("\nX.head():")
display(X.head())
print("\nMissing values per feature:")
print(X.isna().sum().to_string())
print("\nLabel distribution (0=phishing, 1=legitimate):")
print(y.value_counts().sort_index())

Sample URL features (before training):


,URL,URLLength,DomainLength,IsDomainIP,CharContinuationRate,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,ObfuscationRatio,NoOfLettersInURL,LetterRatioInURL,NoOfDegitsInURL,DegitRatioInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfAmpersandInURL,NoOfOtherSpecialCharsInURL,SpacialCharRatioInURL,IsHTTPS
0,https://www.google.com,22,14,0,0.409091,3,1,0,0,0.0,17,0.772727,0,0.0,0,0,0,5,0.227273,1
1,https://example.com,19,11,0,0.526316,3,0,0,0,0.0,15,0.789474,0,0.0,0,0,0,4,0.210526,1
2,http://secure-login-example.com,31,24,0,0.322581,3,0,0,0,0.0,25,0.806452,0,0.0,0,0,0,6,0.193548,0


Captured current model probabilities for before/after comparison.
Extracted 10,000/235,795 URLs
Extracted 20,000/235,795 URLs
Extracted 30,000/235,795 URLs
Extracted 40,000/235,795 URLs
Extracted 50,000/235,795 URLs
Extracted 60,000/235,795 URLs
Extracted 70,000/235,795 URLs
Extracted 80,000/235,795 URLs
Extracted 90,000/235,795 URLs
Extracted 100,000/235,795 URLs
Extracted 110,000/235,795 URLs
Extracted 120,000/235,795 URLs
Extracted 130,000/235,795 URLs
Extracted 140,000/235,795 URLs
Extracted 150,000/235,795 URLs
Extracted 160,000/235,795 URLs
Extracted 170,000/235,795 URLs
Extracted 180,000/235,795 URLs
Extracted 190,000/235,795 URLs
Extracted 200,000/235,795 URLs
Extracted 210,000/235,795 URLs
Extracted 220,000/235,795 URLs
Extracted 230,000/235,795 URLs
Extracted 235,795/235,795 URLs
Successfully processed URLs: 235795
X.shape: (235795, 19)
Feature order verified: True

X.head():


,URLLength,DomainLength,IsDomainIP,CharContinuationRate,TLDLength,NoOfSubDomain,HasObfuscation,NoOfObfuscatedChar,ObfuscationRatio,NoOfLettersInURL,LetterRatioInURL,NoOfDegitsInURL,DegitRatioInURL,NoOfEqualsInURL,NoOfQMarkInURL,NoOfAmpersandInURL,NoOfOtherSpecialCharsInURL,SpacialCharRatioInURL,IsHTTPS
0,32,24,0,0.593750,3,1,0,0,0.0,27,0.843750,0,0.0,0,0,0,5,0.156250,1
1,24,16,0,0.333333,2,1,0,0,0.0,18,0.750000,0,0.0,0,0,0,6,0.250000,1
2,30,22,0,0.500000,2,2,0,0,0.0,24,0.800000,0,0.0,0,0,0,6,0.200000,1
3,27,19,0,0.518519,3,1,0,0,0.0,22,0.814815,0,0.0,0,0,0,5,0.185185,1
4,34,26,0,0.617647,3,1,0,0,0.0,29,0.852941,0,0.0,0,0,0,5,0.147059,1



Missing values per feature:
URLLength                     0
DomainLength                  0
IsDomainIP                    0
CharContinuationRate          0
TLDLength                     0
NoOfSubDomain                 0
HasObfuscation                0
NoOfObfuscatedChar            0
ObfuscationRatio              0
NoOfLettersInURL              0
LetterRatioInURL              0
NoOfDegitsInURL               0
DegitRatioInURL               0
NoOfEqualsInURL               0
NoOfQMarkInURL                0
NoOfAmpersandInURL            0
NoOfOtherSpecialCharsInURL    0
SpacialCharRatioInURL         0
IsHTTPS                       0

Label distribution (0=phishing, 1=legitimate):
label
0    100945
1    134850
Name: count, dtype: int64


In [36]:
# Cell 6 — Group-based train/test split by domain
splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)
group_column = "Domain" if "Domain" in df.columns else "URL"
train_idx, test_idx = next(
    splitter.split(X, y, groups=df[group_column].astype(str))
)
X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()
y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

if set(df.iloc[train_idx][group_column].astype(str)).intersection(
    set(df.iloc[test_idx][group_column].astype(str))
):
    raise AssertionError("A validation group appears in both train and test sets.")
print("Train:", X_train.shape, "Test:", X_test.shape)
print("Grouping column:", group_column)
print("Train groups:", df.iloc[train_idx][group_column].astype(str).nunique())
print("Test groups:", df.iloc[test_idx][group_column].astype(str).nunique())

Train: (189074, 19) Test: (46721, 19)
Grouping column: Domain
Train groups: 176068
Test groups: 44018


In [37]:
# Cell 7 — Train Decision Tree
models = {}
decision_tree = DecisionTreeClassifier(random_state=42)
decision_tree.fit(X_train, y_train)
models["Decision Tree"] = decision_tree
predictions = {"Decision Tree": decision_tree.predict(X_test)}
print("Decision Tree training complete.")

Decision Tree training complete.


In [38]:
# Cell 8 — Train the required Random Forest
random_forest = RandomForestClassifier(
    n_estimators=300,
    max_depth=20,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=42,
    n_jobs=-1,
)
random_forest.fit(X_train, y_train)
models["Random Forest"] = random_forest
predictions["Random Forest"] = random_forest.predict(X_test)
print("Random Forest training complete.")

Random Forest training complete.


In [39]:
# Cell 9 — Train Gaussian Naive Bayes
naive_bayes = GaussianNB()
naive_bayes.fit(X_train, y_train)
models["Naive Bayes"] = naive_bayes
predictions["Naive Bayes"] = naive_bayes.predict(X_test)
print("Naive Bayes training complete.")

Naive Bayes training complete.


In [40]:
# Cell 10 — Train KNN (scale using training data only)
knn = make_pipeline(
    StandardScaler(),
    KNeighborsClassifier(n_neighbors=5, n_jobs=-1),
)
knn.fit(X_train, y_train)
models["KNN"] = knn
predictions["KNN"] = knn.predict(X_test)
print("KNN training complete.")

KNN training complete.


In [50]:
# Cell 11 — Compare all models on the same held-out domains
metrics_by_model = {}
evaluation_matrices = {}
evaluation_reports = {}
comparison_rows = []
for model_name, model_prediction in predictions.items():
    metrics = {
        "Accuracy": float(accuracy_score(y_test, model_prediction)),
        "Precision": float(precision_score(y_test, model_prediction, pos_label=1, zero_division=0)),
        "Recall": float(recall_score(y_test, model_prediction, pos_label=1, zero_division=0)),
        "F1 Score": float(f1_score(y_test, model_prediction, pos_label=1, zero_division=0)),
    }
    metrics_by_model[model_name] = metrics
    evaluation_matrices[model_name] = confusion_matrix(
        y_test, model_prediction, labels=[0, 1]
    ).tolist()
    evaluation_reports[model_name] = classification_report(
        y_test,
        model_prediction,
        labels=[0, 1],
        target_names=["Phishing (0)", "Legitimate (1)"],
        output_dict=True,
        zero_division=0,
    )
    comparison_rows.append({"Model": model_name, **metrics})

comparison_df = pd.DataFrame(
    comparison_rows,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1 Score"],
)
display(comparison_df)

,Model,Accuracy,Precision,Recall,F1 Score
0,Decision Tree,0.994563,0.994805,0.995766,0.995285
1,Random Forest,0.995077,0.995765,0.995691,0.995728
2,Naive Bayes,0.908521,0.866928,0.993797,0.926037
3,KNN,0.992059,0.992032,0.994206,0.993118


In [42]:
# Cell 12 — Confusion matrices and classification reports
class_names = ["Phishing (0)", "Legitimate (1)"]
for model_name, model_prediction in predictions.items():
    print(f"\n{model_name} — confusion matrix (actual rows, predicted columns)")
    matrix = confusion_matrix(y_test, model_prediction, labels=[0, 1])
    display(pd.DataFrame(
        matrix,
        index=[f"Actual {name}" for name in class_names],
        columns=[f"Predicted {name}" for name in class_names],
    ))
    print(f"{model_name} — classification report")
    print(classification_report(
        y_test,
        model_prediction,
        labels=[0, 1],
        target_names=class_names,
        zero_division=0,
    ))


Decision Tree — confusion matrix (actual rows, predicted columns)


,Predicted Phishing (0),Predicted Legitimate (1)
Actual Phishing (0),19658,140
Actual Legitimate (1),114,26809


Decision Tree — classification report
                precision    recall  f1-score   support

  Phishing (0)       0.99      0.99      0.99     19798
Legitimate (1)       0.99      1.00      1.00     26923

      accuracy                           0.99     46721
     macro avg       0.99      0.99      0.99     46721
  weighted avg       0.99      0.99      0.99     46721


Random Forest — confusion matrix (actual rows, predicted columns)


,Predicted Phishing (0),Predicted Legitimate (1)
Actual Phishing (0),19684,114
Actual Legitimate (1),116,26807


Random Forest — classification report
                precision    recall  f1-score   support

  Phishing (0)       0.99      0.99      0.99     19798
Legitimate (1)       1.00      1.00      1.00     26923

      accuracy                           1.00     46721
     macro avg       0.99      0.99      0.99     46721
  weighted avg       1.00      1.00      1.00     46721


Naive Bayes — confusion matrix (actual rows, predicted columns)


,Predicted Phishing (0),Predicted Legitimate (1)
Actual Phishing (0),15691,4107
Actual Legitimate (1),167,26756


Naive Bayes — classification report
                precision    recall  f1-score   support

  Phishing (0)       0.99      0.79      0.88     19798
Legitimate (1)       0.87      0.99      0.93     26923

      accuracy                           0.91     46721
     macro avg       0.93      0.89      0.90     46721
  weighted avg       0.92      0.91      0.91     46721


KNN — confusion matrix (actual rows, predicted columns)


,Predicted Phishing (0),Predicted Legitimate (1)
Actual Phishing (0),19583,215
Actual Legitimate (1),156,26767


KNN — classification report
                precision    recall  f1-score   support

  Phishing (0)       0.99      0.99      0.99     19798
Legitimate (1)       0.99      0.99      0.99     26923

      accuracy                           0.99     46721
     macro avg       0.99      0.99      0.99     46721
  weighted avg       0.99      0.99      0.99     46721



In [43]:
# Cell 13 — Save the trained Random Forest in the app-compatible location
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)
MODEL_FILE = MODELS_DIR / "phishguard_url_model.joblib"
joblib.dump(random_forest, MODEL_FILE)
print("Saved Random Forest:", MODEL_FILE)

Saved Random Forest: c:\Users\vaishu\PhishGuard-AI\models\phishguard_url_model.joblib


In [44]:
# Cell 14 — Save the exact feature order used by training
FEATURE_COLUMNS_FILE = MODELS_DIR / "url_feature_columns.json"
with FEATURE_COLUMNS_FILE.open("w", encoding="utf-8") as feature_file:
    json.dump(URL_FEATURES, feature_file, indent=2)
print("Saved ordered feature list:", FEATURE_COLUMNS_FILE)

Saved ordered feature list: c:\Users\vaishu\PhishGuard-AI\models\url_feature_columns.json


In [45]:
# Cell 15 — Save reproducible training metadata and measured metrics
model_metadata = {
    "project_name": "PhishGuard AI — Outsmarting the Next-Generation Cyber Attack",
    "model_name": "RandomForestClassifier",
    "dataset_name": DATASET_FILE.name,
    "dataset_path": str(DATASET_FILE),
    "target_column": "label",
    "label_mapping": {"0": "phishing", "1": "legitimate"},
    "feature_source": "URL column extracted with url_features.extract_url_features",
    "feature_list": URL_FEATURES,
    "number_of_features": len(URL_FEATURES),
    "urls_successfully_processed": int(len(X)),
    "train_size": int(len(X_train)),
    "test_size": int(len(X_test)),
    "group_column": group_column,
    "validation_method": "GroupShuffleSplit by group column; test_size=0.20",
    "random_state": 42,
    "model_parameters": random_forest.get_params(deep=False),
    "evaluation_metrics": metrics_by_model,
    "confusion_matrices_labels_0_1": evaluation_matrices,
    "classification_reports": evaluation_reports,
    "random_forest_metrics": metrics_by_model["Random Forest"],
    "baseline_sample_predictions_before_retraining": baseline_probabilities,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
}
METADATA_FILE = MODELS_DIR / "model_metadata.json"
with METADATA_FILE.open("w", encoding="utf-8") as metadata_file:
    json.dump(model_metadata, metadata_file, indent=2, default=str)
print("Saved model metadata:", METADATA_FILE)

Saved model metadata: c:\Users\vaishu\PhishGuard-AI\models\model_metadata.json


In [46]:
# Cell 16 — Reload saved model and compare the before/after sample probabilities
loaded_model = joblib.load(MODEL_FILE)
saved_feature_order = json.loads(FEATURE_COLUMNS_FILE.read_text(encoding="utf-8"))
if saved_feature_order != URL_FEATURES:
    raise AssertionError("Saved feature order differs from the training feature order.")
if loaded_model.n_features_in_ != 19:
    raise AssertionError("Reloaded model does not accept exactly 19 features.")

# Recompute samples using the same shared extraction function used for every training row.
final_sample_features = pd.DataFrame(
    [extract_url_features(sample_url) for sample_url in sample_urls],
    columns=saved_feature_order,
)
final_probabilities = loaded_model.predict_proba(final_sample_features)
after_probabilities = {}
comparison_rows = []
for index, sample_url in enumerate(sample_urls):
    class_probabilities = dict(zip(loaded_model.classes_, final_probabilities[index]))
    prediction = int(loaded_model.predict(final_sample_features.iloc[[index]])[0])
    after = {
        "phishing_probability": float(class_probabilities[0]),
        "legitimate_probability": float(class_probabilities[1]),
        "prediction": prediction,
    }
    after_probabilities[sample_url] = after
    before = baseline_probabilities.get(sample_url) if baseline_probabilities else None
    comparison_rows.append({
        "URL": sample_url,
        "Before phishing probability": before["phishing_probability"] if before else np.nan,
        "Before legitimate probability": before["legitimate_probability"] if before else np.nan,
        "After phishing probability": after["phishing_probability"],
        "After legitimate probability": after["legitimate_probability"],
        "Prediction": "phishing" if prediction == 0 else "legitimate",
    })
sample_results = pd.DataFrame(comparison_rows)
print("Reloaded saved model:", MODEL_FILE)
print("Feature order verified:", saved_feature_order == URL_FEATURES)
display(sample_results)
print("URLs successfully processed:", len(X))
print("Feature count:", X.shape[1])
print("Train size:", len(X_train))
print("Test size:", len(X_test))
print("Evaluation metrics:")
display(comparison_df)

Reloaded saved model: c:\Users\vaishu\PhishGuard-AI\models\phishguard_url_model.joblib
Feature order verified: True


,URL,Before phishing probability,Before legitimate probability,After phishing probability,After legitimate probability,Prediction
0,https://www.google.com,0.985196,0.014804,0.005403,0.994597,legitimate
1,https://example.com,0.991635,0.008365,0.982213,0.017787,phishing
2,http://secure-login-example.com,1.000000,0.000000,1.000000,0.000000,phishing


URLs successfully processed: 235795
Feature count: 19
Train size: 189074
Test size: 46721
Evaluation metrics:


,Model,Accuracy,Precision,Recall,F1 Score
0,Decision Tree,0.994563,0.994805,0.995766,0.995285
1,Random Forest,0.995077,0.995765,0.995691,0.995728
2,Naive Bayes,0.908521,0.866928,0.993797,0.926037
3,KNN,0.992059,0.992032,0.994206,0.993118


In [47]:
# Diagnostic: load model from project root

import json
import joblib
import pandas as pd
from pathlib import Path

# Project root
PROJECT_ROOT = Path(r"C:\Users\vaishu\PhishGuard-AI")

# Correct model paths
MODEL_PATH = PROJECT_ROOT / "models" / "phishguard_url_model.joblib"
FEATURE_PATH = PROJECT_ROOT / "models" / "url_feature_columns.json"

# Check files
print("Model path:", MODEL_PATH)
print("Model exists:", MODEL_PATH.exists())

print("Feature file:", FEATURE_PATH)
print("Feature file exists:", FEATURE_PATH.exists())

if not MODEL_PATH.exists():
    raise FileNotFoundError(
        f"Model not found at:\n{MODEL_PATH}\n\n"
        "Run the model training/save cells first."
    )

if not FEATURE_PATH.exists():
    raise FileNotFoundError(
        f"Feature file not found at:\n{FEATURE_PATH}\n\n"
        "Run the model training/save cells first."
    )

# Load model
model = joblib.load(MODEL_PATH)

# Load feature order
with open(FEATURE_PATH, "r", encoding="utf-8") as f:
    url_feature_columns = json.load(f)

print("\n✅ Model loaded successfully!")
print("Number of features:", len(url_feature_columns))
print("Model classes:", model.classes_)

Model path: C:\Users\vaishu\PhishGuard-AI\models\phishguard_url_model.joblib
Model exists: True
Feature file: C:\Users\vaishu\PhishGuard-AI\models\url_feature_columns.json
Feature file exists: True

✅ Model loaded successfully!
Number of features: 19
Model classes: [0 1]


In [48]:
# Cell 28 — Final prediction diagnostic

from url_features import extract_url_features

test_urls = [
    "https://www.google.com",
    "https://example.com",
    "http://secure-login-example.com"
]

for url in test_urls:

    # Extract the same 19 features used by the trained model
    features = extract_url_features(url)

    # Put features into the exact training order
    X_sample = pd.DataFrame(
        [[features[col] for col in url_feature_columns]],
        columns=url_feature_columns
    )

    # Prediction probabilities
    probabilities = model.predict_proba(X_sample)[0]

    # Map probabilities to classes
    class_probabilities = dict(
        zip(model.classes_, probabilities)
    )

    prediction = model.predict(X_sample)[0]

    print("\n" + "=" * 70)
    print("URL:", url)
    print("Phishing probability :", round(class_probabilities[0] * 100, 2), "%")
    print("Legitimate probability:", round(class_probabilities[1] * 100, 2), "%")
    print(
        "Prediction:",
        "PHISHING" if prediction == 0 else "LEGITIMATE"
    )


URL: https://www.google.com
Phishing probability : 0.54 %
Legitimate probability: 99.46 %
Prediction: LEGITIMATE

URL: https://example.com
Phishing probability : 98.22 %
Legitimate probability: 1.78 %
Prediction: PHISHING

URL: http://secure-login-example.com
Phishing probability : 100.0 %
Legitimate probability: 0.0 %
Prediction: PHISHING


In [49]:
# Check the raw URL and label columns

print("URL column exists:", "URL" in df.columns)
print("label column exists:", "label" in df.columns)

print("\nFirst 5 URLs:")
print(df["URL"].head())

print("\nLabel distribution:")
print(df["label"].value_counts())

URL column exists: True
label column exists: True

First 5 URLs:
0      https://www.southbankmosaics.com
1              https://www.uni-mainz.de
2        https://www.voicefmradio.co.uk
3           https://www.sfnmjournal.com
4    https://www.rewildingargentina.org
Name: URL, dtype: str

Label distribution:
label
1    134850
0    100945
Name: count, dtype: int64


In [51]:
# Final validation of the RETRAINED model

from pathlib import Path
import json
import joblib
import pandas as pd
from url_features import extract_url_features

PROJECT_ROOT = Path(r"C:\Users\vaishu\PhishGuard-AI")

MODEL_PATH = PROJECT_ROOT / "models" / "phishguard_url_model.joblib"
FEATURE_PATH = PROJECT_ROOT / "models" / "url_feature_columns.json"

model = joblib.load(MODEL_PATH)

with open(FEATURE_PATH, "r", encoding="utf-8") as f:
    feature_columns = json.load(f)

test_urls = [
    "https://www.google.com",
    "https://example.com",
    "http://secure-login-example.com"
]

print("=" * 75)
print("FINAL RETRAINED MODEL TEST")
print("=" * 75)

for url in test_urls:

    features = extract_url_features(url)

    X_sample = pd.DataFrame(
        [[features[col] for col in feature_columns]],
        columns=feature_columns
    )

    probabilities = model.predict_proba(X_sample)[0]
    classes = model.classes_

    probability_map = dict(zip(classes, probabilities))

    phishing_probability = probability_map.get(0, 0)
    legitimate_probability = probability_map.get(1, 0)

    prediction = model.predict(X_sample)[0]

    print("\nURL:", url)
    print(f"Phishing probability:    {phishing_probability * 100:.2f}%")
    print(f"Legitimate probability: {legitimate_probability * 100:.2f}%")
    print(
        "Prediction:",
        "PHISHING" if prediction == 0 else "LEGITIMATE"
    )

FINAL RETRAINED MODEL TEST

URL: https://www.google.com
Phishing probability:    0.54%
Legitimate probability: 99.46%
Prediction: LEGITIMATE

URL: https://example.com
Phishing probability:    98.22%
Legitimate probability: 1.78%
Prediction: PHISHING

URL: http://secure-login-example.com
Phishing probability:    100.00%
Legitimate probability: 0.00%
Prediction: PHISHING


In [52]:
# FINAL PHISHGUARD AI DEMO TEST — 10 URLs

from pathlib import Path
import json
import joblib
import pandas as pd
from url_features import extract_url_features

# --------------------------------------------------
# 1. Load trained model and feature list
# --------------------------------------------------

PROJECT_ROOT = Path(r"C:\Users\vaishu\PhishGuard-AI")

MODEL_PATH = PROJECT_ROOT / "models" / "phishguard_url_model.joblib"
FEATURE_PATH = PROJECT_ROOT / "models" / "url_feature_columns.json"

model = joblib.load(MODEL_PATH)

with open(FEATURE_PATH, "r", encoding="utf-8") as f:
    feature_columns = json.load(f)

# --------------------------------------------------
# 2. Test URLs
# --------------------------------------------------

test_urls = [
    "https://www.google.com",
    "https://www.microsoft.com",
    "https://www.github.com",
    "https://www.wikipedia.org",
    "https://www.amazon.com",
    "https://example.com",
    "http://secure-login-example.com",
    "http://paypal-login-verification.com",
    "http://account-security-update.com/login",
    "http://verify-your-account-now.com"
]

# --------------------------------------------------
# 3. Prediction function
# --------------------------------------------------

results = []

for url in test_urls:

    features = extract_url_features(url)

    X_sample = pd.DataFrame(
        [[features[col] for col in feature_columns]],
        columns=feature_columns
    )

    probabilities = model.predict_proba(X_sample)[0]

    classes = model.classes_

    probability_map = dict(zip(classes, probabilities))

    phishing_probability = probability_map.get(0, 0)
    legitimate_probability = probability_map.get(1, 0)

    prediction = model.predict(X_sample)[0]

    result = "PHISHING" if prediction == 0 else "LEGITIMATE"

    results.append({
        "URL": url,
        "Phishing Probability (%)": round(phishing_probability * 100, 2),
        "Legitimate Probability (%)": round(legitimate_probability * 100, 2),
        "Prediction": result
    })

# --------------------------------------------------
# 4. Display results
# --------------------------------------------------

results_df = pd.DataFrame(results)

print("=" * 100)
print("PHISHGUARD AI — FINAL URL TEST RESULTS")
print("=" * 100)

print(results_df.to_string(index=False))

print("\n" + "=" * 100)
print("Testing completed successfully.")
print("=" * 100)

PHISHGUARD AI — FINAL URL TEST RESULTS
                                     URL  Phishing Probability (%)  Legitimate Probability (%) Prediction
                  https://www.google.com                      0.54                       99.46 LEGITIMATE
               https://www.microsoft.com                      0.55                       99.45 LEGITIMATE
                  https://www.github.com                      0.54                       99.46 LEGITIMATE
               https://www.wikipedia.org                      0.55                       99.45 LEGITIMATE
                  https://www.amazon.com                      0.54                       99.46 LEGITIMATE
                     https://example.com                     98.22                        1.78   PHISHING
         http://secure-login-example.com                    100.00                        0.00   PHISHING
    http://paypal-login-verification.com                     99.19                        0.81   PHISHING
http://